In [1]:
# ── 3-CLASS HERBARIUM CV UTILITY CLASSIFIER ───────────────────────────────────
# Classifies herbarium images by computer vision utility:
# Class 0: Not useful for CV (in-situ, no visible structures)
# Class 1: Possibly useful but non-typical (wood sections, fragments) 
# Class 2: Typical and useful (standard pressed specimens)

# ── Core imports ───────────────────────────────────────────────────────────────
from pathlib import Path
import os, random, logging, shutil
import pandas as pd
import numpy as np
from PIL import Image

# ── ML imports ─────────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

# ── Configuration ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED         = 1337
IMG_SIZE     = 512
BATCH_SIZE   = 12           # adjust for VRAM
NUM_WORKERS  = 10
PIN_MEMORY   = True
EPOCHS       = 500
PATIENCE     = 20
MIN_DELTA    = 1e-4         # AUC improvement threshold for early stopping
LR           = 5e-5 #3e-6
WARMUP_EPOCHS = 3
LABEL_SMOOTHING = 0.05      # applied manually in train loop

# ── Project structure ──────────────────────────────────────────────────────────
root = Path("/home/john/Desktop/Class/generative segmentation/herbarium_3class_cv")
data_dir = root / "data"
training_dir = data_dir / "training_data"
manifests_dir = data_dir / "manifests"
runs_dir = root / "runs"

# ── Class mapping ──────────────────────────────────────────────────────────────
CLASS_NAMES = {0: "not_useful", 1: "aberrant", 2: "typical"}
NUM_CLASSES = len(CLASS_NAMES)

print(f"Project root: {root}")
print(f"Classes: {CLASS_NAMES}")

Using device: cuda
Project root: /home/john/Desktop/Class/generative segmentation/herbarium_3class_cv
Classes: {0: 'not_useful', 1: 'aberrant', 2: 'typical'}


In [2]:
# ── DATA INGESTION CELL ────────────────────────────────────────────────────────
# Handles two scenarios:
# 1) Ingest new images from inbox folders (with DWCa metadata lookup)
# 2) Create manifests from existing training folders (scanning + DWCa lookup)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ── Config ─────────────────────────────────────────────────────────────────────
DWCA_ROOT = Path("/mnt/slowstorage/26_01_16_plantae_gbif/0014636-260108223611665")
OCC_FP = DWCA_ROOT / "occurrence.txt"
CHUNKSIZE = 500_000

inbox_map = {
    "inbox_class0": training_dir / "class0_not_useful",
    "inbox_class1": training_dir / "class1_aberrant", 
    "inbox_class2": training_dir / "class2_typical",
}
inbox_root = data_dir / "new_examples"

KEEP_OCC = [
    "gbifID", "institutionCode", "collectionCode", "datasetKey",
    "scientificName", "family", "genus", "species", "year", "eventDate",
]

# ── 1. Collect all files that need processing ─────────────────────────────────
# Check inbox folders for new files
inbox_files = {}  # gbifID -> (src_path, dest_dir, class_id)
for folder, dest in inbox_map.items():
    class_id = int(folder.split("_class")[1])
    for fp in (inbox_root / folder).glob("*.jpg"):
        try:
            gid = int(fp.stem)
            inbox_files[gid] = (fp, dest, class_id)
        except ValueError:
            logging.warning("Skipping non-gbifID filename: %s", fp.name)

# Check if we need to build manifests from existing training data
manifest_files = [manifests_dir / f"train_class{i}.csv" for i in range(NUM_CLASSES)]
need_manifest_rebuild = not all(f.exists() for f in manifest_files)

# Collect existing training files if manifests need building
existing_training_files = {}  # gbifID -> (existing_path, class_id)
if need_manifest_rebuild:
    logging.info("Manifest files missing, will scan existing training folders")
    for class_id in range(NUM_CLASSES):
        class_dir = training_dir / f"class{class_id}_{CLASS_NAMES[class_id]}"
        if class_dir.exists():
            for fp in class_dir.glob("*.jpg"):
                try:
                    gid = int(fp.stem)
                    existing_training_files[gid] = (fp, class_id)
                except ValueError:
                    pass

# Combine all files that need metadata lookup
all_needed_ids = set(inbox_files.keys()) | set(existing_training_files.keys())

if inbox_files:
    logging.info("Inbox files to process: %d", len(inbox_files))
if existing_training_files:
    logging.info("Existing training files to catalog: %d", len(existing_training_files))
if not all_needed_ids:
    logging.info("No files to process and manifests exist - nothing to do")

# ── 2. Stream DWCa for metadata if we have files to process ───────────────────
meta_lookup = {}
if all_needed_ids and DWCA_ROOT.exists() and OCC_FP.exists():
    _occ_header = pd.read_csv(OCC_FP, sep="\t", nrows=0).columns.tolist()
    use_cols = [c for c in KEEP_OCC if c in _occ_header]

    logging.info("Streaming occurrence.txt for %d gbifIDs...", len(all_needed_ids))
    occ_parts = []
    for chunk in pd.read_csv(
        OCC_FP, sep="\t", usecols=use_cols,
        chunksize=CHUNKSIZE, low_memory=False,
        dtype={"gbifID": "int64"}, on_bad_lines="skip",
    ):
        sub = chunk[chunk["gbifID"].isin(all_needed_ids)]
        if len(sub):
            occ_parts.append(sub)

    if occ_parts:
        occ_meta = pd.concat(occ_parts, ignore_index=True).drop_duplicates(subset=["gbifID"])
        meta_lookup = occ_meta.set_index("gbifID").to_dict("index")
    
    logging.info("Metadata matched: %d / %d", len(meta_lookup), len(all_needed_ids))
elif all_needed_ids:
    logging.warning("DWCa not found, proceeding without metadata")

# ── 3. Process inbox files (new ingestion) ───────────────────────────────────
new_rows_by_class = {i: [] for i in range(NUM_CLASSES)}
counts = {"ingested": 0, "dedup_skip": 0, "meta_missing": 0, "error": 0}

# Build existing ID set for dedup
existing_ids = set()
for class_id in range(NUM_CLASSES):
    class_dir = training_dir / f"class{class_id}_{CLASS_NAMES[class_id]}"
    if class_dir.exists():
        for fp in class_dir.glob("*.jpg"):
            try:
                existing_ids.add(int(fp.stem))
            except ValueError:
                pass

for gid, (src_path, dest_dir, class_id) in inbox_files.items():
    if gid in existing_ids:
        counts["dedup_skip"] += 1
        continue

    try:
        img = Image.open(src_path).convert("RGB")
        resized = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    except Exception as e:
        logging.warning("Error processing %s: %s", src_path.name, e)
        counts["error"] += 1
        continue

    dest_dir.mkdir(parents=True, exist_ok=True)
    resized.save(dest_dir / src_path.name, format="JPEG", quality=95)
    src_path.unlink()  # Remove original
    counts["ingested"] += 1

    meta = meta_lookup.get(gid, {})
    if not meta:
        counts["meta_missing"] += 1

    base_row = {
        "path": str(dest_dir / src_path.name),
        "filename": src_path.name,
        "gbifID": gid,
        "class_id": class_id,
        "class_name": CLASS_NAMES[class_id],
        "institutionCode": meta.get("institutionCode"),
        "collectionCode": meta.get("collectionCode"),
        "datasetKey": meta.get("datasetKey"),
        "scientificName": meta.get("scientificName"),
        "family": meta.get("family"),
        "genus": meta.get("genus"),
        "species": meta.get("species"),
        "year": meta.get("year"),
        "eventDate": meta.get("eventDate"),
    }
    new_rows_by_class[class_id].append(base_row)

if inbox_files:
    logging.info("Inbox processing — ingested: %d | dedup: %d | meta missing: %d | errors: %d",
                counts["ingested"], counts["dedup_skip"], counts["meta_missing"], counts["error"])

# ── 4. Build manifests from existing training data ───────────────────────────
existing_rows_by_class = {i: [] for i in range(NUM_CLASSES)}
if need_manifest_rebuild:
    logging.info("Building manifests from existing training data...")
    for gid, (existing_path, class_id) in existing_training_files.items():
        meta = meta_lookup.get(gid, {})
        
        base_row = {
            "path": str(existing_path),
            "filename": existing_path.name,
            "gbifID": gid,
            "class_id": class_id,
            "class_name": CLASS_NAMES[class_id],
            "institutionCode": meta.get("institutionCode"),
            "collectionCode": meta.get("collectionCode"),
            "datasetKey": meta.get("datasetKey"),
            "scientificName": meta.get("scientificName"),
            "family": meta.get("family"),
            "genus": meta.get("genus"),
            "species": meta.get("species"),
            "year": meta.get("year"),
            "eventDate": meta.get("eventDate"),
        }
        existing_rows_by_class[class_id].append(base_row)
    
    logging.info("Cataloged existing files: %d", len(existing_training_files))

# ── 5. Write/update manifest CSVs ─────────────────────────────────────────────
manifests_dir.mkdir(parents=True, exist_ok=True)

for class_id in range(NUM_CLASSES):
    csv_fp = manifests_dir / f"train_class{class_id}.csv"
    
    # Combine existing catalog + new ingestion
    all_rows = existing_rows_by_class[class_id] + new_rows_by_class[class_id]
    
    if all_rows:
        df = pd.DataFrame(all_rows)
        df.to_csv(csv_fp, index=False)
        action = "Created" if need_manifest_rebuild else "Updated"
        logging.info("%s %s: %d total rows", action, csv_fp.name, len(df))
    elif need_manifest_rebuild:
        # Create empty manifest for classes with no data
        empty_df = pd.DataFrame(columns=[
            "path", "filename", "gbifID", "class_id", "class_name",
            "institutionCode", "collectionCode", "datasetKey",
            "scientificName", "family", "genus", "species", "year", "eventDate"
        ])
        empty_df.to_csv(csv_fp, index=False)
        logging.info("Created empty manifest: %s", csv_fp.name)

# ── 6. Clean splits if any changes were made ─────────────────────────────────
if inbox_files or need_manifest_rebuild:
    for split_fp in [manifests_dir / f"split_{split}.csv" for split in ["train", "val", "test"]]:
        if split_fp.exists():
            split_fp.unlink()
    logging.info("Cleared split CSVs — Cell 3 will create fresh balanced splits")

INFO: No files to process and manifests exist - nothing to do


In [3]:
# ── DATA LOADING AND SPLITS ────────────────────────────────────────────────────
# Loads class-specific manifests and creates stratified splits with proper class balance
# Automatically detects and fixes imbalanced splits

train_fp = manifests_dir / "split_train.csv"
val_fp   = manifests_dir / "split_val.csv" 
test_fp  = manifests_dir / "split_test.csv"

# Check if existing splits are balanced, regenerate if not
need_regeneration = False
if train_fp.exists() and val_fp.exists() and test_fp.exists():
    try:
        val_check = pd.read_csv(val_fp)
        val_class_dist = val_check["y"].value_counts(normalize=True).sort_index()
        
        # Check if validation set is severely imbalanced (>80% of any class)
        max_class_pct = val_class_dist.max()
        if max_class_pct > 0.8:
            print(f"Detected imbalanced splits (val set {max_class_pct:.1%} single class), regenerating...")
            need_regeneration = True
        else:
            train_df = pd.read_csv(train_fp)
            val_df = pd.read_csv(val_fp)
            test_df = pd.read_csv(test_fp)
            print("Loaded existing balanced splits.")
    except Exception as e:
        print(f"Error reading existing splits: {e}, regenerating...")
        need_regeneration = True
else:
    need_regeneration = True

if need_regeneration:
    # Clean up any corrupted split files
    for fp in [train_fp, val_fp, test_fp]:
        if fp.exists():
            fp.unlink()
    
    # Load class-specific manifests and combine
    class_manifests = {
        0: manifests_dir / "train_class0.csv",     # not_useful
        1: manifests_dir / "train_class1.csv",     # aberrant  
        2: manifests_dir / "train_class2.csv",     # typical
    }
    
    dfs = []
    for class_id, manifest_path in class_manifests.items():
        if manifest_path.exists():
            df = pd.read_csv(manifest_path)
            df["y"] = class_id
            # Use institutionCode for grouping when available, fallback to filename stem
            df["group"] = df.get("institutionCode", df["path"].apply(lambda x: Path(x).stem[:6])).fillna("UNK").astype(str)
            dfs.append(df)
            print(f"Loaded Class {class_id} ({CLASS_NAMES[class_id]}): {len(df)} samples")
        else:
            print(f"Warning: {manifest_path} not found")
    
    if not dfs:
        raise FileNotFoundError("No class manifest files found. Run data ingestion (Cell 2) first.")
    
    # Combine all classes
    df = pd.concat(dfs, ignore_index=True)
    
    print(f"\nTotal samples: {len(df)}")
    print("Class distribution:")
    for class_id in sorted(df["y"].unique()):
        count = (df["y"] == class_id).sum()
        pct = 100 * count / len(df)
        print(f"  Class {class_id} ({CLASS_NAMES[class_id]}): {count} ({pct:.1f}%)")
    
    # Check group diversity
    min_groups_per_class = df.groupby("y")["group"].nunique().min()
    print(f"\nMin groups per class: {min_groups_per_class}")
    
    # Use simple stratification to maintain class proportions
    print("Using simple class stratification to ensure balanced splits")
    from sklearn.model_selection import train_test_split
    
    # 1) Hold out test set (20%)
    trainval_idx, test_idx = train_test_split(
        range(len(df)), 
        test_size=0.2, 
        stratify=df["y"], 
        random_state=SEED
    )
    df_trainval = df.iloc[trainval_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)
    
    # 2) Split trainval into train/val (val ≈ 16% overall)
    train_idx, val_idx = train_test_split(
        range(len(df_trainval)), 
        test_size=0.2, 
        stratify=df_trainval["y"], 
        random_state=SEED + 1
    )
    train_df = df_trainval.iloc[train_idx].reset_index(drop=True)
    val_df = df_trainval.iloc[val_idx].reset_index(drop=True)
    test_df = df_test
    
    # Save splits
    manifests_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(train_fp, index=False)
    val_df.to_csv(val_fp, index=False)
    test_df.to_csv(test_fp, index=False)
    print("Created and saved new balanced splits.")

# ── Split summary ──────────────────────────────────────────────────────────────
def split_report(name, d):
    print(f"\n{name}: n={len(d)}")
    class_counts = d["y"].value_counts().sort_index()
    for class_id, count in class_counts.items():
        pct = 100 * count / len(d)
        print(f"  Class {class_id} ({CLASS_NAMES[class_id]}): {count} ({pct:.1f}%)")
    if "group" in d.columns:
        print(f"  Unique groups: {d['group'].nunique()}")

split_report("TRAIN", train_df)
split_report("VAL", val_df)
split_report("TEST", test_df)

Loaded existing balanced splits.

TRAIN: n=1964
  Class 0 (not_useful): 892 (45.4%)
  Class 1 (aberrant): 313 (15.9%)
  Class 2 (typical): 759 (38.6%)
  Unique groups: 154

VAL: n=491
  Class 0 (not_useful): 223 (45.4%)
  Class 1 (aberrant): 78 (15.9%)
  Class 2 (typical): 190 (38.7%)
  Unique groups: 113

TEST: n=614
  Class 0 (not_useful): 279 (45.4%)
  Class 1 (aberrant): 98 (16.0%)
  Class 2 (typical): 237 (38.6%)
  Unique groups: 115


In [4]:
# ── TRANSFORMS AND DATASET ─────────────────────────────────────────────────────
# 3-class specific transforms and dataset class with error handling

# ── Transforms ─────────────────────────────────────────────────────────────────
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2,
        saturation=0.2, hue=0.05
    ),
    #transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    transforms.RandomErasing(
        p=0.25, scale=(0.02, 0.08),
        ratio=(0.3, 3.3), value="random"
    ),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])

# ── Dataset class ──────────────────────────────────────────────────────────────
class HerbariumDataset(Dataset):
    def __init__(self, df, transform):
        self.paths     = df["path"].tolist()
        self.labels    = df["y"].astype("int64").to_numpy()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            img = self.transform(img)
        except Exception as e:
            print(f"WARNING: could not load {self.paths[idx]}: {e}")
            img = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

# ── Create datasets ────────────────────────────────────────────────────────────
train_ds = HerbariumDataset(train_df, train_tfms)
val_ds   = HerbariumDataset(val_df,   eval_tfms)
test_ds  = HerbariumDataset(test_df,  eval_tfms)

# ── DataLoaders with NO multiprocessing to avoid pickle issues ────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Sanity check ───────────────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"Batch shapes: {xb.shape} | Labels: {yb.shape}")
print(f"Label range: {yb.min().item()} - {yb.max().item()}")
print(f"Sample labels: {yb[:8].tolist()}")

# Check class distribution in training batch
unique, counts = torch.unique(yb, return_counts=True)
print("Batch class distribution:")
for class_id, count in zip(unique.tolist(), counts.tolist()):
    print(f"  Class {class_id} ({CLASS_NAMES[class_id]}): {count}")

Batch shapes: torch.Size([12, 3, 512, 512]) | Labels: torch.Size([12])
Label range: 0 - 2
Sample labels: [0, 0, 0, 2, 2, 0, 0, 0]
Batch class distribution:
  Class 0 (not_useful): 9
  Class 2 (typical): 3


In [5]:
# ── MODEL SETUP ────────────────────────────────────────────────────────────────
# SwinV2-T with 3-class head, class-balanced loss, and mixed precision training

# ── Model architecture ─────────────────────────────────────────────────────────
weights = torchvision.models.Swin_V2_T_Weights.DEFAULT
model   = torchvision.models.swin_v2_t(weights=weights)
model.head = nn.Linear(model.head.in_features, NUM_CLASSES)  # 3-class head
model   = model.to(device)

# ── Loss function with class balancing ─────────────────────────────────────────
# Compute class weights to handle imbalanced datasets automatically
class_counts = train_df["y"].value_counts().sort_index()
total_samples = len(train_df)
class_weights = torch.tensor([
    total_samples / (NUM_CLASSES * count) for count in class_counts
], dtype=torch.float32).to(device)

print(f"Class distribution in training:")
for i, count in enumerate(class_counts):
    weight = class_weights[i].item()
    print(f"  Class {i} ({CLASS_NAMES[i]}): {count} samples, weight: {weight:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── Optimizer ──────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)

# ── Learning rate scheduler ────────────────────────────────────────────────────
warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor = 0.1,
    end_factor   = 1.0,
    total_iters  = WARMUP_EPOCHS,
)
# cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer,
#     T_max  = max(EPOCHS - WARMUP_EPOCHS, 1),
#     eta_min= LR * 0.01,
# )

cosine_sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,           # Restart every 10 epochs
    T_mult=1,         # Keep T constant
    eta_min=LR * 0.1,
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers = [warmup_sched, cosine_sched],
    milestones = [WARMUP_EPOCHS],
)

# ── Mixed precision scaler ─────────────────────────────────────────────────────
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ── Sanity forward pass ────────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
xb = xb.to(device, non_blocking=True)
yb = yb.to(device, non_blocking=True)

with torch.no_grad():
    with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
        logits = model(xb)
        loss = criterion(logits, yb)

print(f"Forward pass: logits {logits.shape}, loss: {loss.item():.4f}")
print(f"Logits sample: {logits[0].detach().cpu().numpy()}")

Class distribution in training:
  Class 0 (not_useful): 892 samples, weight: 0.734
  Class 1 (aberrant): 313 samples, weight: 2.092
  Class 2 (typical): 759 samples, weight: 0.863


/tmp/ipykernel_142454/2716051332.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Forward pass: logits torch.Size([12, 3]), loss: 1.0174
Logits sample: [ 0.666   -0.4365  -0.04523]


In [6]:
# ── TRAIN/EVAL LOOPS ───────────────────────────────────────────────────────────
# 3-class training with label smoothing, mixed precision, and comprehensive metrics

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Label smoothing for 3-class
        yb_smooth = torch.zeros(yb.size(0), NUM_CLASSES, device=device)
        yb_smooth.fill_(LABEL_SMOOTHING / NUM_CLASSES)
        yb_smooth.scatter_(1, yb.unsqueeze(1), 1.0 - LABEL_SMOOTHING + LABEL_SMOOTHING / NUM_CLASSES)

        optimizer.zero_grad(set_to_none=True)

        # Fixed autocast call
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss = F.cross_entropy(logits, yb_smooth)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * xb.size(0)
        y_true.append(yb.detach().cpu().numpy())
        y_pred.append(torch.argmax(logits, dim=1).detach().cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    return {
        "loss": running_loss / len(loader.dataset),
        "acc": accuracy_score(y_true, y_pred),
        "cm": confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)),
    }


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    y_true, y_pred, y_probs = [], [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Fixed autocast call
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss = criterion(logits, yb)

        running_loss += loss.item() * xb.size(0)
        y_true.append(yb.detach().cpu().numpy())
        y_pred.append(torch.argmax(logits, dim=1).detach().cpu().numpy())
        y_probs.append(torch.softmax(logits, dim=1).detach().cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_probs = np.concatenate(y_probs)

    return {
        "loss": running_loss / len(loader.dataset),
        "acc": accuracy_score(y_true, y_pred),
        "cm": confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)),
        "probs": y_probs,
    }


# ── Metrics helper ─────────────────────────────────────────────────────────────
def print_metrics(name, metrics, class_names=CLASS_NAMES):
    """Print comprehensive metrics for 3-class problem"""
    print(f"\n{name}:")
    print(f"  Loss: {metrics['loss']:.5f} | Accuracy: {metrics['acc']:.4f}")
    
    # Confusion matrix with class names
    cm = metrics['cm']
    print("  Confusion Matrix:")
    print("    " + "".join(f"{class_names[i]:>12}" for i in range(NUM_CLASSES)))
    for i in range(NUM_CLASSES):
        row_str = f"{class_names[i]:>8} "
        row_str += "".join(f"{cm[i, j]:>12}" for j in range(NUM_CLASSES))
        print(row_str)
    
    # Per-class precision/recall if available
    if len(np.unique(metrics['cm'])) > 1:
        for i in range(NUM_CLASSES):
            tp = cm[i, i]
            fp = cm[:, i].sum() - tp
            fn = cm[i, :].sum() - tp
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            
            print(f"  {class_names[i]:>12}: P={precision:.3f} R={recall:.3f}")

In [7]:
# ── TRAINING LOOP ──────────────────────────────────────────────────────────────
# 3-class training with early stopping on validation accuracy

save_dir = runs_dir / "swinv2t_3class"
save_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = save_dir / "best.pt"

history = []
best_val_acc = -1.0
best_epoch = 0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_metrics = eval_one_epoch(model, val_loader, criterion, device)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
    }
    history.append(row)

    improved = val_metrics["acc"] > best_val_acc + MIN_DELTA
    if improved:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch
        bad_epochs = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_val_acc": best_val_acc,
            "config": {
                "arch": "swin_v2_t",
                "num_classes": NUM_CLASSES,
                "img_size": IMG_SIZE,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "seed": SEED,
                "class_weights": class_weights.cpu().tolist(),
            },
        }, ckpt_path)
        tag = "✓ saved"
    else:
        bad_epochs += 1
        tag = f"patience {bad_epochs}/{PATIENCE}"

    print(
        f"epoch {epoch:02d} | lr {current_lr:.2e} | "
        f"train loss {row['train_loss']:.4f} acc {row['train_acc']:.3f} | "
        f"val loss {row['val_loss']:.4f} acc {row['val_acc']:.3f} | "
        f"{tag}"
    )

    # Only show detailed metrics every 10 epochs or when best
    #if epoch % 10 == 0 or improved:
    if epoch % 10 == 0:  # reduce spam a bit.
        print_metrics("TRAIN", train_metrics)
        print_metrics("VAL", val_metrics)

    if bad_epochs >= PATIENCE:
        print(f"\nEarly stopping — epoch {epoch} | "
              f"best epoch {best_epoch} | best val_acc {best_val_acc:.4f}")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(save_dir / "history.csv", index=False)
print(f"\nTraining complete. Best checkpoint: {ckpt_path}")
print(f"Best epoch: {best_epoch} | Best val_acc: {best_val_acc:.4f}")

epoch 01 | lr 2.00e-05 | train loss 0.6675 acc 0.761 | val loss 0.4611 acc 0.892 | ✓ saved
epoch 02 | lr 3.50e-05 | train loss 0.3697 acc 0.915 | val loss 0.2414 acc 0.941 | ✓ saved
epoch 03 | lr 5.00e-05 | train loss 0.3171 acc 0.936 | val loss 0.2460 acc 0.949 | ✓ saved
epoch 04 | lr 4.89e-05 | train loss 0.2871 acc 0.950 | val loss 0.2530 acc 0.947 | patience 1/20
epoch 05 | lr 4.57e-05 | train loss 0.2646 acc 0.961 | val loss 0.2079 acc 0.945 | patience 2/20
epoch 06 | lr 4.07e-05 | train loss 0.2426 acc 0.968 | val loss 0.1877 acc 0.963 | ✓ saved
epoch 07 | lr 3.45e-05 | train loss 0.2388 acc 0.975 | val loss 0.2032 acc 0.959 | patience 1/20
epoch 08 | lr 2.75e-05 | train loss 0.2129 acc 0.982 | val loss 0.1494 acc 0.961 | patience 2/20
epoch 09 | lr 2.05e-05 | train loss 0.2057 acc 0.984 | val loss 0.1664 acc 0.967 | ✓ saved
epoch 10 | lr 1.43e-05 | train loss 0.1949 acc 0.990 | val loss 0.1833 acc 0.961 | patience 1/20

TRAIN:
  Loss: 0.19489 | Accuracy: 0.9903
  Confusion Matri

In [8]:
# ── FINAL EVALUATION ───────────────────────────────────────────────────────────
# Load best checkpoint and evaluate with detailed metrics

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

val_metrics = eval_one_epoch(model, val_loader, criterion, device)
test_metrics = eval_one_epoch(model, test_loader, criterion, device)

print(f"Best checkpoint — epoch: {ckpt['epoch']} | val_acc: {ckpt['best_val_acc']:.6f}")
print_metrics("VALIDATION", val_metrics)
print_metrics("TEST", test_metrics)

# Generate test predictions for detailed analysis
y_test_true = []
y_test_pred = []
y_test_probs = []

model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(xb)
        
        probs = torch.softmax(logits, dim=1)
        
        y_test_true.extend(yb.cpu().numpy())
        y_test_pred.extend(torch.argmax(logits, dim=1).cpu().numpy())
        y_test_probs.extend(probs.cpu().numpy())

print("\nDetailed Test Set Classification Report:")
print(classification_report(
    y_test_true, y_test_pred, 
    target_names=[f"Class {i} ({CLASS_NAMES[i]})" for i in range(NUM_CLASSES)],
    digits=4
))

# ── Save inference model ───────────────────────────────────────────────────────
inference_path = save_dir / "model_infer.pt"
torch.save({
    "arch": "swin_v2_t",
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "state_dict": model.state_dict(),
    "img_size": IMG_SIZE,
    "norm_mean": (0.485, 0.456, 0.406),
    "norm_std": (0.229, 0.224, 0.225),
    "val_acc": val_metrics["acc"],
    "test_acc": test_metrics["acc"],
    "class_weights": class_weights.cpu().tolist(),
    "best_epoch": ckpt["epoch"],
    "training_completed": True,
}, inference_path)

print(f"\nInference checkpoint saved: {inference_path}")

# ── Save predictions for analysis ──────────────────────────────────────────────
test_predictions = pd.DataFrame({
    "path": test_df["path"],
    "true_class": y_test_true,
    "pred_class": y_test_pred,
    "true_name": [CLASS_NAMES[c] for c in y_test_true],
    "pred_name": [CLASS_NAMES[c] for c in y_test_pred],
    "correct": [t == p for t, p in zip(y_test_true, y_test_pred)],
})

# Add prediction probabilities
y_test_probs = np.array(y_test_probs)
for i in range(NUM_CLASSES):
    test_predictions[f"prob_class{i}"] = y_test_probs[:, i]

test_predictions.to_csv(save_dir / "test_predictions.csv", index=False)
print(f"Test predictions saved: {save_dir / 'test_predictions.csv'}")

print(f"\n3-Class Herbarium CV Classifier complete!")
print(f"Final test accuracy: {test_metrics['acc']:.4f}")
print(f"Model ready for deployment at: {inference_path}")

Best checkpoint — epoch: 21 | val_acc: 0.969450

VALIDATION:
  Loss: 0.18217 | Accuracy: 0.9695
  Confusion Matrix:
      not_useful    aberrant     typical
not_useful          219           3           1
aberrant            1          70           7
 typical            0           3         187
    not_useful: P=0.995 R=0.982
      aberrant: P=0.921 R=0.897
       typical: P=0.959 R=0.984

TEST:
  Loss: 0.19671 | Accuracy: 0.9625
  Confusion Matrix:
      not_useful    aberrant     typical
not_useful          278           1           0
aberrant            5          87           6
 typical            0          11         226
    not_useful: P=0.982 R=0.996
      aberrant: P=0.879 R=0.888
       typical: P=0.974 R=0.954

Detailed Test Set Classification Report:
                      precision    recall  f1-score   support

Class 0 (not_useful)     0.9823    0.9964    0.9893       279
  Class 1 (aberrant)     0.8788    0.8878    0.8832        98
   Class 2 (typical)     0.9741    0.95

In [9]:
# ── STANDALONE HERBARIUM CV CLASSIFIER INFERENCE ──────────────────────────────
# Minimal, self-contained inference code for 3-class herbarium image classification
# Copy this cell into other projects for easy integration

import torch
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from pathlib import Path

class HerbariumClassifier:
    """
    3-class herbarium image classifier for computer vision utility assessment.
    
    Classes:
        0: Not useful for CV (field photos, no visible structures, closed packets)
        1: Possibly useful but aberrant (wood sections, fragments, non-standard)
        2: Typical and useful (standard pressed specimens with clear features)
    """
    
    def __init__(self, model_path=None, device=None):
        # Default to the trained model path
        if model_path is None:
            model_path = "/home/john/Desktop/Class/generative segmentation/herbarium_3class_cv/runs/swinv2t_3class/model_infer.pt"
        
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Load model checkpoint
        checkpoint = torch.load(model_path, map_location=self.device)
        self.config = checkpoint
        self.class_names = checkpoint["class_names"]
        
        # Initialize model architecture
        self.model = models.swin_v2_t(weights=None)
        self.model.head = torch.nn.Linear(self.model.head.in_features, checkpoint["num_classes"])
        self.model.load_state_dict(checkpoint["state_dict"])
        self.model = self.model.to(self.device)
        self.model.eval()
        
        # Setup preprocessing
        self.transform = transforms.Compose([
            transforms.Resize((checkpoint["img_size"], checkpoint["img_size"])),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=checkpoint["norm_mean"],
                std=checkpoint["norm_std"]
            ),
        ])
        
        print(f"HerbariumClassifier loaded: {checkpoint['arch']} | "
              f"Test acc: {checkpoint.get('test_acc', 'N/A'):.3f}")
    
    def classify_image(self, image_path):
        """
        Classify a single herbarium image.
        
        Args:
            image_path: str or Path to image file
            
        Returns:
            dict: {
                'class_id': int,
                'class_name': str, 
                'confidence': float,
                'probabilities': [float, float, float]
            }
        """
        # Load and preprocess image
        image = Image.open(image_path).convert("RGB")
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        # Run inference
        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = self.model(input_tensor)
            
            probabilities = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
            class_id = int(probabilities.argmax())
            confidence = float(probabilities[class_id])
        
        return {
            'class_id': class_id,
            'class_name': self.class_names[class_id],
            'confidence': confidence,
            'probabilities': probabilities.tolist()
        }
    
    def classify_batch(self, image_paths, batch_size=32):
        """
        Classify multiple images efficiently.
        
        Args:
            image_paths: list of str/Path to image files
            batch_size: int, processing batch size
            
        Returns:
            list of classification dicts (same format as classify_image)
        """
        results = []
        
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]
            batch_tensors = []
            
            # Load batch
            for path in batch_paths:
                try:
                    image = Image.open(path).convert("RGB")
                    tensor = self.transform(image)
                    batch_tensors.append(tensor)
                except Exception as e:
                    print(f"Error loading {path}: {e}")
                    # Add dummy tensor for failed loads
                    batch_tensors.append(torch.zeros(3, self.config["img_size"], self.config["img_size"]))
            
            # Run inference on batch
            batch_input = torch.stack(batch_tensors).to(self.device)
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    logits = self.model(batch_input)
                
                probabilities = F.softmax(logits, dim=1).cpu().numpy()
            
            # Format results
            for j, probs in enumerate(probabilities):
                class_id = int(probs.argmax())
                results.append({
                    'class_id': class_id,
                    'class_name': self.class_names[class_id],
                    'confidence': float(probs[class_id]),
                    'probabilities': probs.tolist()
                })
        
        return results


# ── USAGE EXAMPLES ─────────────────────────────────────────────────────────────

# Initialize classifier (uses default model path)
classifier = HerbariumClassifier()

# Single image classification example
# result = classifier.classify_image("path/to/herbarium/image.jpg")
# print(f"Classification: Class {result['class_id']} ({result['class_name']}) "
#       f"with {result['confidence']:.3f} confidence")

# Batch processing example
# image_list = ["image1.jpg", "image2.jpg", "image3.jpg"]
# batch_results = classifier.classify_batch(image_list)
# for i, result in enumerate(batch_results):
#     print(f"{image_list[i]}: {result['class_name']} ({result['confidence']:.3f})")

# Integration function: filter GBIF download
def filter_herbarium_images(image_directory, min_confidence=0.8):
    """Example: Filter directory of herbarium images, keep only useful ones."""
    image_paths = list(Path(image_directory).glob("*.jpg"))
    results = classifier.classify_batch([str(p) for p in image_paths])
    
    useful_images = []
    for path, result in zip(image_paths, results):
        # Keep Class 1 (aberrant) and Class 2 (typical) with high confidence
        if result['class_id'] in [1, 2] and result['confidence'] >= min_confidence:
            useful_images.append((path, result))
    
    print(f"Filtered {len(image_paths)} images → {len(useful_images)} useful for CV")
    return useful_images

print("HerbariumClassifier ready for use!")
print("Available methods: classify_image(path), classify_batch(paths)")
print("Example filter function: filter_herbarium_images(directory)")

HerbariumClassifier loaded: swin_v2_t | Test acc: 0.963
HerbariumClassifier ready for use!
Available methods: classify_image(path), classify_batch(paths)
Example filter function: filter_herbarium_images(directory)


In [10]:

ex_paths = ["/home/john/Desktop/Class/generative segmentation/Magnoliaceae_pipeline/images/rejected_exsitu/1503310753.jpg",
            "/mnt/storage/Magnoliaceae_Segmentation/Magnolia_dataset/images/1929466604.jpg",
            "/mnt/storage/Myrtaceae_Segmentation/myrteae_dataset/images/1424074306.jpg",
            "/mnt/storage/synthetic_dataset/9993_src_0000.jpg",
            "/home/john/Desktop/Class/generative segmentation/situ_classifier/data/insitu_raw/584574606.jpg",
            "/home/john/Desktop/specimen_source.png",
            "/home/john/Desktop/fish_source.png",
            "/home/john/Desktop/Props_ss.png",
            "/home/john/Desktop/Class/generative segmentation/LLM_captions/test_captions/QA_rejected/2859014606.jpg"]

for ex_path in ex_paths:
    result = classifier.classify_image(ex_path)
    print("=======================")
    print(ex_path)
    print(f"Classification: Class {result['class_id']} ({result['class_name']}) "
          f"with {result['confidence']:.3f} confidence")
    print(result)


/home/john/Desktop/Class/generative segmentation/Magnoliaceae_pipeline/images/rejected_exsitu/1503310753.jpg
Classification: Class 2 (typical) with 0.963 confidence
{'class_id': 2, 'class_name': 'typical', 'confidence': 0.962890625, 'probabilities': [0.0172576904296875, 0.0196380615234375, 0.962890625]}
/mnt/storage/Magnoliaceae_Segmentation/Magnolia_dataset/images/1929466604.jpg
Classification: Class 2 (typical) with 0.947 confidence
{'class_id': 2, 'class_name': 'typical', 'confidence': 0.947265625, 'probabilities': [0.0218658447265625, 0.031097412109375, 0.947265625]}
/mnt/storage/Myrtaceae_Segmentation/myrteae_dataset/images/1424074306.jpg
Classification: Class 2 (typical) with 0.972 confidence
{'class_id': 2, 'class_name': 'typical', 'confidence': 0.9716796875, 'probabilities': [0.01554107666015625, 0.01261138916015625, 0.9716796875]}
/mnt/storage/synthetic_dataset/9993_src_0000.jpg
Classification: Class 2 (typical) with 0.970 confidence
{'class_id': 2, 'class_name': 'typical', 'c